# import dependencies

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import spacy
import pandas as pd

# Load and prepare Dataset

In [ ]:
urdu_data = pd.read_pickle('data.pkl')
urdu_data

In [ ]:
test_data = urdu_data[900:]
urdu_data = urdu_data[0:900]

In [ ]:
# Use spaCy for sentence tokenization
nlp = spacy.blank('ur')
nlp.add_pipe('sentencizer')

In [ ]:
def tokenize_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]
    
# Tokenize all documents into sentences
urdu_data['sentences'] = urdu_data['Text'].apply(tokenize_sentences)

In [ ]:
# Flatten all sentences into a single list to build vocabulary and sequence data
all_sentences = [sent for sublist in urdu_data['sentences'] for sent in sublist]

In [ ]:
# Initialize and fit the tokenizer
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(all_sentences)

In [ ]:
sentence_sequences = [tokenizer.texts_to_sequences(sents) for sents in urdu_data['sentences']]

In [ ]:
max_length = max(len(seq) for sublist in sentence_sequences for seq in sublist)
padded_sequences = [pad_sequences(sublist, maxlen=max_length, padding='post') for sublist in sentence_sequences]

In [ ]:
# Create a flat list of sequences and a corresponding label list
X = np.concatenate(padded_sequences)
labels = []

# Create labels based on inclusion in the short summary
for idx, row in urdu_data.iterrows():
    summary_sentences = tokenize_sentences(row['Short Summary'])
    document_sentences = row['sentences']
    labels += [1 if any(sen in doc for sen in summary_sentences) else 0 for doc in document_sentences]

y = np.array(labels)

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
max_length

# Build and train model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional, Dropout

# Model configuration
embedding_dim = 100

# Building the model
model = Sequential([
    Embedding(input_dim=5000, output_dim=embedding_dim, input_length=max_length),
    Bidirectional(LSTM(64, return_sequences=False)),  # Return sequences to get output for each input
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Training the model
model.fit(X_train, y_train, epochs=10, validation_split=0.2)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")
model.save('LSTM_Summarizer.h5')

# Test model

In [ ]:
import tensorflow as tf 
model = tf.keras.models.load_model('LSTM_Summarizer.h5')

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)

# Make predictions
predictions = model.predict(X_test)

In [ ]:
def generate_summary(text, model, tokenizer, max_length):
    # Tokenize text
    nlp = spacy.blank('ur')  # Load the appropriate language model
    nlp.add_pipe('sentencizer')
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    # Convert sentences to sequences and pad them
    sequences = tokenizer.texts_to_sequences(sentences)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

    # Get predictions from the model
    predictions = model.predict(padded_sequences)
    predicted_sentences = [s for s, p in zip(sentences, predictions) if p > 0.5]

    # Combine the selected sentences to form a summary
    return ' '.join(predicted_sentences)

# Example usage
new_text = urdu_data['Text'][510]
summary = generate_summary(new_text, model, tokenizer,  max_length)
print('Text',new_text)
print("Predicted Summary:", summary)
print('Orignal Summary',urdu_data['Short Summary'][510])


In [ ]:
test_data

In [ ]:
orig = []
pred = []
for i in range(900,999)
    summary = generate_summary(test_data['Text'][i], model, tokenizer,  max_length)
    orig.append(test_data['Short Summary'][i])

ValueError: too many values to unpack (expected 2)

In [ ]:
from sklearn.metrics import classification_report

# Predict on test data
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Generate classification report
report = classification_report(y_test, y_pred, target_names=['Not in Summary', 'In Summary'])
print(report)

In [ ]:
from rouge_score import rouge_scorer
import numpy as np  # Import numpy for averaging

def calculate_rouge_scores(references, hypotheses):
    # Initialize the RougeScorer with the desired metrics and a stemmer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = {
        'rouge1': [],
        'rouge2': [],
        'rougeL': []
    }

    # Score each pair of reference and hypothesis
    for ref, hyp in zip(references, hypotheses):
        score = scorer.score(ref, hyp)
        scores['rouge1'].append(score['rouge1'].fmeasure)
        scores['rouge2'].append(score['rouge2'].fmeasure)
        scores['rougeL'].append(score['rougeL'].fmeasure)

    # Calculate average scores using numpy to average the lists
    avg_scores = {key: np.mean(val) for key, val in scores.items()}
    return avg_scores
